# Semi-Supervised SVM (S3VM / TSVM) - Starter Notebook
S3VM and TSVM use unlabeled points to push decision boundaries into low-density regions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [ ]:
# Nonlinear dataset with scarce labels
X, y = make_moons(n_samples=1600, noise=0.22, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rng = np.random.RandomState(42)
idx = np.arange(len(X_train))
rng.shuffle(idx)
labeled_count = 120
labeled_idx = idx[:labeled_count]
unlab_idx = idx[labeled_count:]

X_l, y_l = X_train[labeled_idx], y_train[labeled_idx]
X_u = X_train[unlab_idx]
print('Labeled:', len(X_l), 'Unlabeled:', len(X_u))

In [ ]:
# Supervised SVM baseline
svm_sup = SVC(kernel='rbf', gamma='scale', C=1.0, random_state=42)
svm_sup.fit(X_l, y_l)
acc_sup = accuracy_score(y_test, svm_sup.predict(X_test))

# TSVM-like simple self-training approximation
proba_model = SVC(kernel='rbf', gamma='scale', C=1.0, probability=True, random_state=42)
proba_model.fit(X_l, y_l)
u_prob = proba_model.predict_proba(X_u)
u_conf = u_prob.max(axis=1)
u_label = u_prob.argmax(axis=1)

high_conf = u_conf > 0.92
X_aug = np.vstack([X_l, X_u[high_conf]])
y_aug = np.hstack([y_l, u_label[high_conf]])

svm_ssl = SVC(kernel='rbf', gamma='scale', C=1.0, random_state=42)
svm_ssl.fit(X_aug, y_aug)
acc_ssl = accuracy_score(y_test, svm_ssl.predict(X_test))

print(f'Supervised SVM accuracy: {acc_sup:.4f}')
print(f'TSVM-like SSL accuracy : {acc_ssl:.4f}')
print('Pseudo-labeled used   :', np.sum(high_conf))

In [ ]:
# Decision boundary comparison
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

Z_sup = svm_sup.predict(grid).reshape(xx.shape)
Z_ssl = svm_ssl.predict(grid).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
axes[0].contourf(xx, yy, Z_sup, alpha=0.3, cmap='coolwarm')
axes[0].scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=12, cmap='coolwarm', edgecolors='k')
axes[0].set_title(f'Supervised SVM (acc={acc_sup:.3f})')

axes[1].contourf(xx, yy, Z_ssl, alpha=0.3, cmap='coolwarm')
axes[1].scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=12, cmap='coolwarm', edgecolors='k')
axes[1].set_title(f'TSVM-like SSL SVM (acc={acc_ssl:.3f})')

for ax in axes:
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')

plt.tight_layout()
plt.show()